# Phase 0 — Reference genome check & HPO extraction

1. Confirm GRCh37 vs GRCh38 from the VCF header (`##reference`, chr1 length).
2. Parse the proband's HPO terms from `Challenge_Clinical_Phenotype_1.docx`.

Outputs: `data/work/phase0/00_reference_check.txt`, `data/work/phase0/00_hpo_terms.tsv`

In [ ]:
import re

import docx
import pandas as pd
import pysam

from rare_disease_mva import config as C

C.PHASE0_DIR.mkdir(parents=True, exist_ok=True)

## 1. Reference genome
chr1 length: GRCh38 = 248,956,422 / GRCh37 = 249,250,621.

In [ ]:
vcf = pysam.VariantFile(str(C.RAW_VCF))
refs = [str(r).strip() for r in vcf.header.records if r.key == "reference"]
contigs = vcf.header.contigs
chr1_name = "1" if "1" in contigs else "chr1"
chr1_len = contigs[chr1_name].length

if chr1_len == C.GRCH38_CHR1_LENGTH:
    build = "GRCh38"
elif chr1_len == C.GRCH37_CHR1_LENGTH:
    build = "GRCh37"
else:
    build = f"UNKNOWN (chr1 length {chr1_len})"

report = "\n".join([
    f"Build (from chr1 length {chr1_len:,}): {build}",
    f"Contig naming: {'no chr prefix (1, 2, ..., X, Y, MT)' if chr1_name == '1' else 'chr-prefixed'}",
    f"Total contigs in header: {len(contigs)}",
    f"Samples: {list(vcf.header.samples)}",
    "##reference lines:",
    *[f"  {r}" for r in refs],
    "",
    "Verdict: no liftover needed." if build == "GRCh38" else "Verdict: liftover to GRCh38 required.",
])
(C.PHASE0_DIR / "00_reference_check.txt").write_text(report + "\n")
print(report)

## 2. HPO terms from the clinical phenotype docx
Table header: Clinical Feature | HPO Term | HPO ID | Presentation / Notes

In [ ]:
doc = docx.Document(str(C.PHENOTYPE_DOCX))
rows = []
for table in doc.tables:
    for row in table.rows[1:]:
        cells = [c.text.strip() for c in row.cells]
        if len(cells) >= 3 and re.fullmatch(r"HP:\d{7}", cells[2]):
            rows.append({"clinical_feature": cells[0], "hpo_term": cells[1], "hpo_id": cells[2]})

hpo = pd.DataFrame(rows).drop_duplicates("hpo_id")
hpo.to_csv(C.PHASE0_DIR / "00_hpo_terms.tsv", sep="\t", index=False)
print(f"{len(hpo)} HPO terms")
hpo